# Danh gia 2-Stage Pipeline (Test Evaluate)

Notebook nay chi thuc hien **DANH GIA** (khong train), yeu cau:
- Da upload model weights len Kaggle Dataset
- Da chuan bi du lieu test

### Cac buoc:
1. Cai dat
2. Chuan bi du lieu test
3. Danh gia pipeline 2-Stage_FINAL (evaluate_2stage.py, conf=0.40)
4. Danh gia pipeline 2-Stage_SAHI (evaluate_2stage.py --sahi, conf=0.35)
5. Zip ket qua

In [1]:
# ============================================================
# 1. Tai Ma nguon & Cai dat thu vien
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q sahi ultralytics timm


Cloning into 'waste-detection2-Stage'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 302 (delta 11), reused 25 (delta 8), pack-reused 273 (from 2)
Receiving objects: 100% (302/302), 202.89 MiB | 43.82 MiB/s, done.
Resolving deltas: 100% (134/134), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cud

In [2]:
# ============================================================
# 2. Nap Du lieu Ngoai lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Da tai: {os.path.basename(task['src'])}")
    else:
        print(f"Bo qua: {task['src']} (Khong tim thay tren Kaggle Dataset)")

Da tai: dataset-resized
Da tai: data
Da tai: RealWaste


In [3]:
# ============================================================
# 3. Chuan bi du lieu test 
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("--- Tao nhan YOLO & Split dataset ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("\nHoan tat chuan bi du lieu test!")


/kaggle/working/waste-detection2-Stage
--- Tao nhan YOLO & Split dataset ---
  DATA CLEANING - TACO Dataset

[Load] annotations.json ...
  So anh goc         : 1500
  So annotations goc : 4784
  So categories      : 60

────────────────────────────────────────────────────────────
[Buoc 1] Kiem tra dong trung lap (Duplicates)
────────────────────────────────────────────────────────────
  Duplicate annotations: 0
  Duplicate image IDs: 0
  Duplicate file names: 0

────────────────────────────────────────────────────────────
[Buoc 2] Kiem tra gia tri thieu (Missing Values)
────────────────────────────────────────────────────────────
  Images: Tat ca truong bat buoc day du
  Annotations: Tat ca truong bat buoc day du
  Anh khong co annotation: 0

────────────────────────────────────────────────────────────
[Buoc 3] Kiem tra nhan khong hop le
────────────────────────────────────────────────────────────
  Annotations voi category_id khong hop le: 0
  Categories khong co trong mapping.csv: 1


In [10]:
# ============================================================
# 4. Danh gia Toan Trinh - Ban 2-Stage_FINAL
#    Model    : models/final_best.pt + models/final_best.pth
#    conf     : 0.40 (nguong toi uu)
#    Macro F1 : tinh tren 5 lop rac (khong tinh Background)
# ============================================================

%cd /kaggle/working/waste-detection2-Stage

!python src/evaluate_2stage.py \
    --detector models/2-Stage_best.pt \
    --classifier models/2-Stage_best.pth \
    --data-dir data/processed/images/val \
    --label-dir data/processed/labels/val \
    --conf 0.4 \
    --output results/eval_final_v2


/kaggle/working/waste-detection2-Stage
[INFO] Đang sử dụng thiết bị: cuda
[INFO] Đã load classifier weights từ: models/2-Stage_best.pth
[INFO] num_classes = 6, classes = ['Background', 'Glass', 'Metal', 'Other', 'Paper', 'Plastic']
[INFO] Bắt đầu đánh giá trên 228 ảnh test...
Evaluating: 100%|█████████████████████████████| 228/228 [00:35<00:00,  6.48it/s]

[DEBUG] Tổng GT boxes    : 713
[DEBUG] Tổng Pred boxes   : 583
[DEBUG] IoU matches (≥0.50): 438
[DEBUG] Tỷ lệ GT được match : 61.4%

  KẾT QUẢ ĐÁNH GIÁ (PER CLASS)
            Precision  Recall      F1
Background     0.0000  0.0000  0.0000
Glass          0.5200  0.5652  0.5417
Metal          0.5566  0.8194  0.6629
Other          0.4481  0.3080  0.3651
Paper          0.5972  0.5375  0.5658
Plastic        0.7257  0.5223  0.6074
--------------------------------------------------
  Macro Precision : 0.5695  (trên 5 lớp rác, không tính Background)
  Macro Recall    : 0.5505  (trên 5 lớp rác, không tính Background)
  Macro F1 (≈mAP) : 0.54

In [5]:
# ============================================================
# 5. Danh gia Toan Trinh - Ban 2-Stage_SAHI
#    Model    : stage1_tiled_best.pt + models/final_best.pth
#    conf     : 0.35 (SAHI can nguong thap hon de giu recall)
#    SAHI     : slice=512, overlap=0.2
#    Macro F1 : tinh tren 5 lop rac (khong tinh Background)
# ============================================================

%cd /kaggle/working/waste-detection2-Stage

!python src/evaluate_2stage.py \
    --detector models/2-Stage_SAHI.pt \
    --classifier models/2-Stage_SAHI.pth \
    --data-dir data/processed/images/test \
    --label-dir data/processed/labels/test \
    --conf 0.35 \
    --sahi \
    --output results/eval_sahi_v2


/kaggle/working/waste-detection2-Stage
[INFO] Đang sử dụng thiết bị: cuda
[INFO] Đang load SAHI AutoDetectionModel...
[INFO] Đã load classifier weights từ: models/2-Stage_SAHI.pth
[INFO] num_classes = 6, classes = ['Background', 'Glass', 'Metal', 'Other', 'Paper', 'Plastic']
[INFO] Bắt đầu đánh giá trên 217 ảnh test...
Evaluating: 100%|█████████████████████████████| 217/217 [03:46<00:00,  1.05s/it]

[DEBUG] Tổng GT boxes    : 763
[DEBUG] Tổng Pred boxes   : 1244
[DEBUG] IoU matches (≥0.50): 350
[DEBUG] Tỷ lệ GT được match : 45.9%

  KẾT QUẢ ĐÁNH GIÁ (PER CLASS)
            Precision  Recall      F1
Background     0.0000  0.0000  0.0000
Glass          0.1081  0.1951  0.1391
Metal          0.3169  0.5556  0.4036
Other          0.2037  0.3929  0.2683
Paper          0.1827  0.5373  0.2727
Plastic        0.2657  0.3029  0.2830
--------------------------------------------------
  Macro Precision : 0.2154  (trên 5 lớp rác, không tính Background)
  Macro Recall    : 0.3967  (trên 5 lớp rác, kh

In [6]:
# ============================================================
# 6. Zip ket qua de tai ve
# ============================================================
import shutil

shutil.make_archive("/kaggle/working/eval_results", "zip",
                    "/kaggle/working/waste-detection2-Stage/results")
print("Da nen ket qua: /kaggle/working/eval_results.zip")


Da nen ket qua: /kaggle/working/eval_results.zip
